# 01 · ADMET EDA
엔드포인트별 분포(회귀 히스토그램 / 분류 불균형), 데이터 크기, TDC vs MoleculeNet 비교, 물성 상관, 대표 분자.

> 신약 파이프라인 시리즈 2/4 — Tox21(분류) → **ADMET(회귀+분류)** → 생성 → 결합.

In [ ]:
# --- Colab 자가치유 셋업 (로컬/CI에서는 자동 스킵) ---
import os, sys, subprocess
try:
    import src  # repo 루트에서 실행 중이면 바로 import
except ModuleNotFoundError:
    if not os.path.exists("admet-property-predictor"):
        subprocess.run(["git", "clone",
            "https://github.com/zqvo04/admet-property-predictor.git"], check=False)
    os.chdir("admet-property-predictor")
    subprocess.run(["bash", "setup_colab.sh"], check=False)
    import src
print("src", src.__version__, "| cwd", os.getcwd())

In [ ]:
# === CONFIG (Colab에서 값만 키우면 전체 재현) ===
# 검증용 기본값은 작게 설정 — 실제 학습 시 EPOCHS↑, 엔드포인트 추가.
import os
TDC_PATH = "data/"
EPOCHS   = int(os.environ.get("ADMET_EPOCHS", 2))   # Colab 실전: 100
REG_EP   = ["Caco2"]          # 회귀 엔드포인트 (확장: Solubility, Lipophilicity, ...)
CLS_EP   = ["hERG"]           # 분류 엔드포인트 (확장: BBBP, CYP3A4_Inhibition, ...)
ENDPOINTS_USED = REG_EP + CLS_EP
SEED = 1
print("endpoints:", ENDPOINTS_USED, "| epochs:", EPOCHS)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from src.data import ENDPOINTS, REG_ENDPOINTS, CLS_ENDPOINTS, load_tdc_endpoint
sns.set_context('notebook'); sns.set_style('whitegrid')
print('회귀:', REG_ENDPOINTS)
print('분류:', CLS_ENDPOINTS)
pd.DataFrame(ENDPOINTS).T[['type','source','metric','group']]

## 데이터 로딩 + 크기

In [ ]:
data = {}
for ep in ENDPOINTS_USED:
    data[ep] = load_tdc_endpoint(ep, seed=SEED, tdc_path=TDC_PATH)
    print(data[ep])
sizes = pd.DataFrame({ep: {'train': len(d.train), 'valid': len(d.valid),
                          'test': len(d.test), 'type': d.task_type}
                      for ep, d in data.items()}).T
sizes

## 회귀 타깃 분포 (원단위)

In [ ]:
reg_used = [e for e in ENDPOINTS_USED if ENDPOINTS[e]['type']=='reg']
if reg_used:
    fig, axes = plt.subplots(1, len(reg_used), figsize=(5*len(reg_used),4), squeeze=False)
    for ax, ep in zip(axes[0], reg_used):
        sns.histplot(data[ep].train['Y'], kde=True, ax=ax)
        ax.set_title(f'{ep} (TDC, metric={ENDPOINTS[ep]["metric"]})')
    plt.tight_layout(); plt.show()
else:
    print('no regression endpoint in CONFIG')

## 분류 클래스 불균형

In [ ]:
cls_used = [e for e in ENDPOINTS_USED if ENDPOINTS[e]['type']=='cls']
for ep in cls_used:
    vc = data[ep].train['Y'].value_counts(normalize=True).sort_index()
    print(f'{ep}: pos rate = {vc.get(1.0, 0):.3f}  (n={len(data[ep].train)})')

## TDC vs MoleculeNet 교차검증 (동일 엔드포인트, 출처별 라벨링)

In [ ]:
from src.data import load_moleculenet
try:
    mn = load_moleculenet('bbbp', seed=0)
    print('MoleculeNet BBBP:', mn)
    print('  pos rate:', round(float(mn.train['Y'].mean()),3), '(source=moleculenet)')
    print('주의: TDC BBB_Martins와 분자 수/정의가 다를 수 있어 출처를 명확히 라벨링')
except Exception as e:
    print('MoleculeNet 로딩 스킵(네트워크):', e)

## 물성 상관 (ECFP 차원축소 대신 RDKit descriptor 간단 상관)

In [ ]:
from rdkit.Chem import Descriptors, Crippen
from src.featurizers import smiles_to_mol
ep0 = ENDPOINTS_USED[0]
rows = []
for smi, y in zip(data[ep0].train['Drug'][:300], data[ep0].train['Y'][:300]):
    m = smiles_to_mol(smi)
    if m is None: continue
    rows.append({'Y': y, 'MW': Descriptors.MolWt(m), 'LogP': Crippen.MolLogP(m),
                 'TPSA': Descriptors.TPSA(m), 'HBD': Descriptors.NumHDonors(m),
                 'HBA': Descriptors.NumHAcceptors(m)})
desc = pd.DataFrame(rows)
plt.figure(figsize=(5,4)); sns.heatmap(desc.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title(f'{ep0}: 물성-타깃 상관'); plt.tight_layout(); plt.show()

## 대표 분자

In [ ]:
from rdkit.Chem import Draw
from src.featurizers import smiles_to_mol
mols = [smiles_to_mol(s) for s in data[ENDPOINTS_USED[0]].train['Drug'][:6]]
Draw.MolsToGridImage([m for m in mols if m], molsPerRow=3, subImgSize=(200,150))

✅ **PHASE/EDA 요약**: 데이터 크기·분포·불균형·출처(TDC vs MoleculeNet)·물성 상관·대표 분자 확인 완료. 다음: `02_ECFP_Baselines`.